# 02 — Cleaning & Preprocessing

**Objetivo**: tomar el dataset crudo y producir un parquet limpio que sirva
de entrada estable a los notebooks de feature engineering y modelado.

**Inputs**: parquets raw en `$RAW_DIR` (los mismos que el notebook 01).

**Outputs**:
- `datasets/processed/cicids_clean.parquet`: dataset post-cleaning (gitignored).
- Decisiones documentadas para reproducibilidad.

**Decisiones que se aplican acá**:

1. Drop de columnas con varianza 0 (constantes globales).
2. Reemplazo de `Inf` por `NaN` y luego imputación con 0 (las únicas Inf están
   en features de tasa cuando `Flow Duration = 0`).
3. Fix de encoding en `Label` (`–` corrupto en Web Attacks).
4. Mapeo de las 15 etiquetas originales a **6 categorías** finales.
5. Deduplicación a nivel de fila completa.
6. Normalización de nombres de columna a `snake_case` con `_per_s`.
7. Persistencia como parquet.

In [1]:
import os
import re
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

RAW_DIR = os.environ.get(
    'RAW_DIR',
    os.path.abspath(os.path.join(os.getcwd(), '..', 'datasets', 'raw'))
)
PROCESSED_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'datasets', 'processed'))
os.makedirs(PROCESSED_DIR, exist_ok=True)
OUT_PATH = os.path.join(PROCESSED_DIR, 'cicids_clean.parquet')
print(f"RAW_DIR       = {RAW_DIR}")
print(f"PROCESSED_DIR = {PROCESSED_DIR}")

RAW_DIR       = /home/keppler-exe/Documentos/Maestria Proyecto Capston/ProyectoMIA/obj1_requisitos_datos/data/raw
PROCESSED_DIR = /run/media/keppler-exe/KINGSTON/Laboratorio-MLCyber/datasets/processed


## 1. Carga + concat

Mismo proceso que el notebook 01 — agregamos columna `Day` para preservar la
procedencia temporal (necesaria para el split estratificado posterior).

In [2]:
files = sorted(f for f in os.listdir(RAW_DIR) if f.endswith('.parquet'))
frames = []
for f in files:
    d = pd.read_parquet(os.path.join(RAW_DIR, f))
    d['Day'] = f.split('-no-metadata')[0]
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print(f"Shape inicial: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Shape inicial: (2313810, 79)
Memory usage: 631.4 MB


## 2. Drop de columnas constantes

Las columnas con varianza 0 no aportan a la clasificación (todas las filas tienen
el mismo valor). Las identificamos y dropeamos. Son típicamente flags que
CICFlowMeter calcula pero que nunca se activan en el dataset (Bwd PSH/URG flags,
contadores de bulk-mode no usados, etc.).

In [3]:
numeric = df.select_dtypes(include=[np.number]).columns.tolist()
stds = df[numeric].std()
constant_cols = stds[stds == 0].index.tolist()
print(f"Columnas constantes ({len(constant_cols)}):")
for c in constant_cols:
    print(f"  - {c} (valor = {df[c].iloc[0]})")
df = df.drop(columns=constant_cols)
print(f"\nShape post-drop: {df.shape}")

Columnas constantes (8):
  - Bwd PSH Flags (valor = 0)
  - Bwd URG Flags (valor = 0)
  - Fwd Avg Bytes/Bulk (valor = 0)
  - Fwd Avg Packets/Bulk (valor = 0)
  - Fwd Avg Bulk Rate (valor = 0)
  - Bwd Avg Bytes/Bulk (valor = 0)
  - Bwd Avg Packets/Bulk (valor = 0)
  - Bwd Avg Bulk Rate (valor = 0)

Shape post-drop: (2313810, 71)


## 3. Inf y NaN

Reemplazamos `±Inf` por `NaN` (más fáciles de filtrar/imputar) y luego
imputamos `NaN` con `0`. Esta es la convención típica para CICFlowMeter:
los Inf vienen de divisiones por duración cero, y el modelo aprende mejor
con un valor sentinela explícito que con filas removidas (perderíamos
ataques minoritarios).

In [4]:
before_inf = df.replace([np.inf, -np.inf], np.nan).isna().sum().sum()
df = df.replace([np.inf, -np.inf], np.nan)
print(f"Celdas con NaN/Inf antes de imputar: {before_inf:,}")

# Imputar NaN con 0 solo en numéricas
num_now = df.select_dtypes(include=[np.number]).columns
df[num_now] = df[num_now].fillna(0.0)
print(f"NaN remanentes (en columnas numéricas): {df[num_now].isna().sum().sum()}")

Celdas con NaN/Inf antes de imputar: 0
NaN remanentes (en columnas numéricas): 0


## 4. Fix de encoding en `Label`

CICIDS2017 original usa el caracter `–` (en-dash, U+2013) para separar
"Web Attack" del subtipo. Algunos parquets lo guardaron mal y aparece como
`�`. Lo normalizamos a `-` (guion ASCII).

In [5]:
print("Labels originales (antes de fix):")
for lab, c in df['Label'].value_counts().items():
    repr_lab = repr(lab)
    print(f"  {repr_lab:50s} {c:>10,}")


Labels originales (antes de fix):
  'Benign'                                            1,977,318
  'DoS Hulk'                                            172,846
  'DDoS'                                                128,014
  'DoS GoldenEye'                                        10,286
  'FTP-Patator'                                           5,931
  'DoS slowloris'                                         5,385
  'DoS Slowhttptest'                                      5,228
  'SSH-Patator'                                           3,219
  'PortScan'                                              1,956
  'Web Attack � Brute Force'                              1,470
  'Bot'                                                   1,437
  'Web Attack � XSS'                                        652
  'Infiltration'                                             36
  'Web Attack � Sql Injection'                               21
  'Heartbleed'                                               11


In [6]:
# Reemplazar SOLO los caracteres no-ASCII por ' - ' (los guiones ASCII
# existentes en 'FTP-Patator', 'SSH-Patator' se mantienen intactos).
def normalize_label(s):
    if not isinstance(s, str):
        return s
    # Caracteres no imprimibles ASCII → ' - '
    s = re.sub(r'[^\x20-\x7e]+', ' - ', s)
    # Colapsar múltiples espacios
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df['Label'] = df['Label'].apply(normalize_label)
print("Labels post-normalización:")
for lab, c in df['Label'].value_counts().items():
    print(f"  {lab!r:55s} {c:>10,}")

Labels post-normalización:
  'Benign'                                                 1,977,318
  'DoS Hulk'                                                 172,846
  'DDoS'                                                     128,014
  'DoS GoldenEye'                                             10,286
  'FTP-Patator'                                                5,931
  'DoS slowloris'                                              5,385
  'DoS Slowhttptest'                                           5,228
  'SSH-Patator'                                                3,219
  'PortScan'                                                   1,956
  'Web Attack - Brute Force'                                   1,470
  'Bot'                                                        1,437
  'Web Attack - XSS'                                             652
  'Infiltration'                                                  36
  'Web Attack - Sql Injection'                                    21
  'Hear

## 5. Mapeo a 6 categorías

CICIDS2017 trae 15 etiquetas distintas (Benign, DDoS, 5 variantes de DoS,
2 de Brute Force, 3 de Web Attack, PortScan, Bot, Infiltration). Para que
el modelo sea interpretable y las clases tengan tamaño razonable las
reagrupamos en **6 categorías**:

| Original | → | Categoría final |
|---|---|---|
| `Benign` | → | `Benign` |
| `DDoS` | → | `DDoS` |
| `DoS Hulk`, `DoS GoldenEye`, `DoS Slowhttptest`, `DoS slowloris`, `Heartbleed` | → | `DoS` |
| `FTP-Patator`, `SSH-Patator` | → | `Brute Force` |
| `Web Attack - XSS`, `Web Attack - Sql Injection`, `Web Attack - Brute Force` | → | `Web Attack` |
| `PortScan`, `Bot`, `Infiltration` | → | `Reconnaissance` |

`Reconnaissance` agrupa actividades de descubrimiento/lateralización (port
scanning, botnets que hacen recon, infiltraciones) — son escasas
individualmente pero conceptualmente similares.

In [7]:
LABEL_MAP = {
    'Benign': 'Benign',
    'DDoS': 'DDoS',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'DoS slowloris': 'DoS',
    'Heartbleed': 'DoS',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'Web Attack - XSS': 'Web Attack',
    'Web Attack - Sql Injection': 'Web Attack',
    'Web Attack - Brute Force': 'Web Attack',
    'PortScan': 'Reconnaissance',
    'Bot': 'Reconnaissance',
    'Infiltration': 'Reconnaissance',
}

df['Label_6'] = df['Label'].map(LABEL_MAP)
unmapped = df[df['Label_6'].isna()]['Label'].unique()
print(f"Labels sin mapeo: {len(unmapped)}")
for lab in unmapped:
    print(f"  - {lab!r}")

Labels sin mapeo: 0


In [8]:
# Si quedan unmapped, descartar (o agregar al mapping). Acá descartamos.
n_before = len(df)
df = df[df['Label_6'].notna()].copy()
print(f"Filas descartadas por label sin mapeo: {n_before - len(df):,}")
print(f"\nDistribución Label_6:")
print(df['Label_6'].value_counts())

Filas descartadas por label sin mapeo: 0

Distribución Label_6:
Label_6
Benign            1977318
DoS                193756
DDoS               128014
Brute Force          9150
Reconnaissance       3429
Web Attack           2143
Name: count, dtype: int64


## 6. Deduplicación

CICFlowMeter puede generar flujos duplicados cuando hay reinicios del sniffer
o cuando un mismo flujo se procesa múltiples veces en distintas ventanas. La
bitácora del proyecto reportó ~28% de duplicados.

Dedupeamos a nivel de **fila completa** (todas las features + Label_6
idénticas). Si dos flujos son bit-exact iguales, conservar uno solo no pierde
información.

In [9]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
removed = before - after
pct = removed / before * 100
print(f"Filas antes:    {before:>10,}")
print(f"Filas después:  {after:>10,}")
print(f"Duplicados:     {removed:>10,} ({pct:.2f}%)")

Filas antes:     2,313,810
Filas después:   2,313,810
Duplicados:              0 (0.00%)


## 7. Normalización de nombres de columnas

CICIDS2017 usa nombres con espacios y barras (`Flow Bytes/s`,
`Total Fwd Packets`). Para evitar quoting/escaping en código posterior,
los pasamos a `snake_case` con la convención `/s → _per_s`.

In [10]:
def norm_col(c):
    c = c.strip().replace('/s', '_per_s').replace('/', '_per_')
    return re.sub(r'\s+', '_', c)

new_names = {c: norm_col(c) for c in df.columns}
df = df.rename(columns=new_names)
print("Renombres aplicados (primeros 15):")
for old, new in list(new_names.items())[:15]:
    if old != new:
        print(f"  {old:35s} → {new}")
print(f"\n... total {sum(1 for o,n in new_names.items() if o != n)} columnas renombradas.")

Renombres aplicados (primeros 15):
  Flow Duration                       → Flow_Duration
  Total Fwd Packets                   → Total_Fwd_Packets
  Total Backward Packets              → Total_Backward_Packets
  Fwd Packets Length Total            → Fwd_Packets_Length_Total
  Bwd Packets Length Total            → Bwd_Packets_Length_Total
  Fwd Packet Length Max               → Fwd_Packet_Length_Max
  Fwd Packet Length Min               → Fwd_Packet_Length_Min
  Fwd Packet Length Mean              → Fwd_Packet_Length_Mean
  Fwd Packet Length Std               → Fwd_Packet_Length_Std
  Bwd Packet Length Max               → Bwd_Packet_Length_Max
  Bwd Packet Length Min               → Bwd_Packet_Length_Min
  Bwd Packet Length Mean              → Bwd_Packet_Length_Mean
  Bwd Packet Length Std               → Bwd_Packet_Length_Std
  Flow Bytes/s                        → Flow_Bytes_per_s

... total 68 columnas renombradas.


## 8. Persistencia

Guardamos el parquet limpio en `datasets/processed/cicids_clean.parquet` para
que los notebooks 03+ lo consuman sin re-ejecutar todo este pipeline.

In [11]:
df.to_parquet(OUT_PATH, index=False)
size_mb = os.path.getsize(OUT_PATH) / 1024**2
print(f"Escrito {OUT_PATH}")
print(f"Tamaño:  {size_mb:.1f} MB")
print(f"Shape:   {df.shape}")
print(f"\nPrimeras 5 columnas: {list(df.columns)[:5]}")
print(f"Últimas  3 columnas: {list(df.columns)[-3:]}")

Escrito /run/media/keppler-exe/KINGSTON/Laboratorio-MLCyber/datasets/processed/cicids_clean.parquet
Tamaño:  249.2 MB
Shape:   (2313810, 72)

Primeras 5 columnas: ['Protocol', 'Flow_Duration', 'Total_Fwd_Packets', 'Total_Backward_Packets', 'Fwd_Packets_Length_Total']
Últimas  3 columnas: ['Label', 'Day', 'Label_6']


## 9. Conclusiones

**Lo que produjo este notebook**:

- Un parquet `cicids_clean.parquet` con shape final post-dedup, sin columnas
  constantes, sin Inf, con `Label_6` ya mapeado a 6 categorías y nombres
  de columna en `snake_case`.
- Una columna `Day` adicional para preservar la información temporal (la usa
  el split estratificado del notebook 04).

**Lo que NO hicimos (deliberadamente)**:

- **Drop de features por leakage o redundancia**: eso va en el notebook 03
  (feature audit con VIF y análisis de leakage).
- **Scaling/normalización de features**: lo hace el `StandardScaler` que se
  fitea en el notebook 06 sobre el train split (evitamos data leakage entre
  train y test).
- **Balanceo de clases**: depende del modelo. Algunos manejan imbalance
  con `class_weight='balanced'`, otros requieren oversampling. Decisión
  postergada al notebook 04/05.

**Próximo notebook (03)**: feature audit — identificar columnas con leakage
(features que solo se conocen DESPUÉS de la decisión, ej: `Flow Bytes/s`
calculado al final del flujo) y reducir colinealidad con VIF.